In [2]:
import pandas as pd
import spacy
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# Lade deine CSV-Datei
data = pd.read_csv('C:/Users/doehr/Documents/GitHub/Project_supply_chain/supply_chain_project_trustpilot_advanced_merge.csv')

# Vorschau
data.head()



TypeError: ForwardRef._evaluate() missing 1 required keyword-only argument: 'recursive_guard'

In [5]:
# Bereinige HTML-ähnlichen Text in Kommentaren
def extract_comment(text):
    if isinstance(text, str):
        matches = re.findall(r'>([^<]+)<', text)
        if matches:
            return matches[0]
    return text  # Wenn kein String oder kein Treffer, gib den Text unverändert zurück

# Wende die Bereinigung an
data['Comment'] = data['Comment'].apply(extract_comment)

# Füge Überschrift und Kommentar zusammen
data['Text'] = data['Heading'].fillna('') + ' ' + data['Comment'].fillna('')

# Entferne ungenutzte Spalten
data = data.drop(['Name', 'Heading', 'Comment'], axis=1)

# Vorschau
data.head()



,Unnamed: 0,Company,Rating_number_customer,Stars,Invitation,Dates,Text
0,0,skatedeluxe,2,5,Auf Einladung,5. März 2025,"Jederzeit wieder Sehr schnelle Lieferung, gute..."
1,1,skatedeluxe,2,5,Auf Einladung,5. März 2025,Schnelle Lieferung No comment
2,2,skatedeluxe,1,5,Auf Einladung,4. März 2025,Bester Service und top Qualität Der bestellvor...
3,3,skatedeluxe,1,5,Auf Einladung,4. März 2025,Schnelligkeit Ausgefallene Produkte
4,4,skatedeluxe,2,5,Auf Einladung,3. März 2025,"Super Service Super Service, extrem schnelle L..."


In [7]:
# Lade das deutsche Sprachmodell
nlp = spacy.load('de_core_news_sm')

# Tokenizer + Lemmatizer kombiniert
def preprocess_text(text):
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
    return tokens

# Füge tokenisierten und lemmatisierten Text hinzu
data['tokens'] = data['Text'].apply(preprocess_text)

# Nur für TF-IDF später, wieder in einen durch Leerzeichen getrennten Text umwandeln
data['clean_text'] = data['tokens'].apply(lambda x: ' '.join(x))

# Vorschau
data.head()



,Unnamed: 0,Company,Rating_number_customer,Stars,Invitation,Dates,Text,tokens,clean_text
0,0,skatedeluxe,2,5,Auf Einladung,5. März 2025,"Jederzeit wieder Sehr schnelle Lieferung, gute...","[Jederzeit, schnell, Lieferung, Preis-Leistung...",Jederzeit schnell Lieferung Preis-Leistungsver...
1,1,skatedeluxe,2,5,Auf Einladung,5. März 2025,Schnelle Lieferung No comment,"[schnell, Lieferung, --, comment]",schnell Lieferung -- comment
2,2,skatedeluxe,1,5,Auf Einladung,4. März 2025,Bester Service und top Qualität Der bestellvor...,"[Bester, Service, Top, Qualität, Bestellvorgan...",Bester Service Top Qualität Bestellvorgang unk...
3,3,skatedeluxe,1,5,Auf Einladung,4. März 2025,Schnelligkeit Ausgefallene Produkte,"[Schnelligkeit, Ausgefallene, Produkt]",Schnelligkeit Ausgefallene Produkt
4,4,skatedeluxe,2,5,Auf Einladung,3. März 2025,"Super Service Super Service, extrem schnelle L...","[Super, Service, Super, Service, extrem, schne...",Super Service Super Service extrem schnell Lie...


In [2]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import RandomOverSampler

In [ ]:
# Trenne Merkmale und Labels
X = data['clean_text']
y = data['Stars']

# Train-Test-Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Vektorisierer
vec = TfidfVectorizer()

# Auf Training anpassen, beide transformieren
X_train_vec = vec.fit_transform(X_train)
X_test_vec = vec.transform(X_test)

#oversampling
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X_train_vec, y_train)

# Modeltesting: Which model has the best accuracy?

models = {
    'Gradient Boosting': GradientBoostingClassifier(),
    'Random Forest': RandomForestClassifier(),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier()
}

for name, model in models.items():
    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_test_vec)
    print(f'{name} Accuracy: {accuracy_score(y_test, y_pred)}')
    print(classification_report(y_test, y_pred))

NameError: name 'data' is not defined

In [ ]:
# RandomForest deliver nearly the same accuracy of 0.86!

In [10]:
# instead of over- or undersampling I choosed the BalanceRandomForestClassifier. Perhaps we also could do a test with oversampling or SMOTE

from imblearn.ensemble import BalancedRandomForestClassifier

bclf = BalancedRandomForestClassifier()
bclf.fit(X_train_vec, y_train) 
y_pred = bclf.predict(X_test_vec)
print(" Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

 Accuracy: 0.7924374319912949
              precision    recall  f1-score   support

           1       0.81      0.87      0.84       966
           2       0.68      0.50      0.58       159
           3       0.60      0.56      0.58       192
           4       0.27      0.50      0.35       237
           5       0.93      0.83      0.88      2122

    accuracy                           0.79      3676
   macro avg       0.66      0.65      0.65      3676
weighted avg       0.83      0.79      0.80      3676



In [ ]:
# The accuracy of these resampling-method is not so good as without resampling!
# accuracy - Randomforest : 0.868
# accuracy - BalancedRandomForestClassifier: 0.79